In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy
import random
import os  # <--- THIS WAS MISSING
from tqdm import tqdm

# ==========================================
# 1. UTILS: MIXUP, EMA, & METRICS
# ==========================================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

def get_device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_class_subset(dataset, classes):
    indices = [i for i, label in enumerate(dataset.targets) if label in classes]
    return Subset(dataset, indices)

def mixup_data(x, y, alpha=0.4): # Low alpha for stability
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.register(model)

    def register(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data = self.shadow[name]

def cosine_distill_loss(student_proj, teacher_proj):
    loss = 0
    num_shared = len(teacher_proj)
    for i in range(num_shared):
        s = student_proj[i].flatten(1)
        t = teacher_proj[i].flatten(1)
        loss += (1 - F.cosine_similarity(s, t, dim=1)).mean()
    return loss

def build_replay_buffer(dataset, classes, per_class=1000): # High Capacity
    indices = []
    counts = {k:0 for k in classes}
    for i, label in enumerate(dataset.targets):
        if label in classes and counts[label] < per_class:
            indices.append(i)
            counts[label] += 1
    return Subset(dataset, indices)

# ==========================================
# 2. MODEL ARCHITECTURE (RESNET-LITE)
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False), nn.BatchNorm2d(out_c))
    def forward(self, x): return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        # ResNet-18 Style (Deeper & Wider) for NMI performance
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.final_conv = nn.Conv2d(256, 64, 1) # Project back to 64
        
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = []
        layers.append(ResidualBlock(in_c, out_c, stride))
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = self.final_conv(x)
        return x

class Broker(nn.Module):
    def __init__(self, n_specs, ch, n_classes):
        super().__init__()
        self.projections = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(n_specs)])
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(n_specs*ch, n_specs*ch//4, 1), nn.ReLU(), 
            nn.Conv2d(n_specs*ch//4, n_specs*ch, 1), nn.Sigmoid()
        )
        self.spatial_gate = nn.Sequential(nn.Conv2d(2, 1, 7, 1, 3), nn.Sigmoid())
        self.bottleneck = nn.Sequential(nn.Conv2d(n_specs*ch, ch, 1), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(ch, n_classes)

    def forward(self, feats):
        feats = [F.interpolate(f, size=(4,4)) for f in feats]
        proj = [p(f) for p, f in zip(self.projections, feats)]
        z_raw = torch.cat(proj, 1)
        w_c = self.channel_gate(z_raw)
        z = z_raw * w_c
        avg_out = torch.mean(z, 1, keepdim=True)
        max_out, _ = torch.max(z, 1, keepdim=True)
        w_s = self.spatial_gate(torch.cat([avg_out, max_out], 1))
        z = z * w_s
        out = self.bottleneck(z)
        logits = self.classifier(self.pool(out).flatten(1))
        
        # Logit Bias (Friend's Fix)
        if logits.shape[1] > 5:
            logits[:, :5] *= 1.1 
        return logits, proj

class KFN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist(), Specialist()])
        self.broker = Broker(2, 64, n_classes)
    def forward(self, x): return self.broker([s(x) for s in self.specialists])

# SURGICAL EXPANSION
def expand_classifier(old_layer, new_num_classes):
    old_out, in_features = old_layer.weight.shape
    new_layer = nn.Linear(in_features, new_num_classes)
    with torch.no_grad():
        new_layer.weight[:old_out] = old_layer.weight
        new_layer.bias[:old_out] = old_layer.bias
    return new_layer

def expand_conv_layer(old_layer, new_in_channels):
    out_ch, old_in_ch, k, _ = old_layer.weight.shape
    new_layer = nn.Conv2d(new_in_channels, out_ch, kernel_size=k, stride=old_layer.stride, padding=old_layer.padding)
    with torch.no_grad():
        new_layer.weight[:, :old_in_ch, :, :] = old_layer.weight
        new_layer.bias[:] = old_layer.bias
        nn.init.kaiming_normal_(new_layer.weight[:, old_in_ch:, :, :], nonlinearity='relu')
    return new_layer

def expand_channel_gate(old_gate, num_old, num_new, ch_per_spec):
    old_in = num_old * ch_per_spec
    new_in = (num_old + num_new) * ch_per_spec
    old_hidden = old_in // 4
    new_hidden = new_in // 4
    old_conv1, old_conv2 = old_gate[1], old_gate[3]
    new_conv1 = nn.Conv2d(new_in, new_hidden, 1)
    new_conv2 = nn.Conv2d(new_hidden, new_in, 1)
    with torch.no_grad():
        new_conv1.weight[:old_hidden, :old_in] = old_conv1.weight
        new_conv1.bias[:old_hidden] = old_conv1.bias
        nn.init.kaiming_normal_(new_conv1.weight[old_hidden:, :])
        nn.init.kaiming_normal_(new_conv1.weight[:, old_in:])
        new_conv2.weight[:old_in, :old_hidden] = old_conv2.weight
        new_conv2.bias[:old_in] = old_conv2.bias
        nn.init.kaiming_normal_(new_conv2.weight[old_in:, :])
        nn.init.kaiming_normal_(new_conv2.weight[:, old_hidden:])
    return nn.Sequential(nn.AdaptiveAvgPool2d(1), new_conv1, nn.ReLU(), new_conv2, nn.Sigmoid())

# ==========================================
# 3. EXPERIMENT EXECUTION (SAFE PLATINUM)
# ==========================================
def run():
    set_deterministic(42)
    device = get_device()
    os.makedirs("logs_safe", exist_ok=True)
    
    # Data Setup
    stats = ((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    train_T = transforms.Compose([
        transforms.RandomCrop(32, 4), transforms.RandomHorizontalFlip(), 
        transforms.ToTensor(), transforms.Normalize(*stats)
    ])
    test_T = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])
    
    full_train = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_T)
    full_test = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_T)
    
    loader_A = DataLoader(get_class_subset(full_train, [0,1,2,3,4]), 128, shuffle=True, num_workers=2)
    loader_B = DataLoader(get_class_subset(full_train, [5,6,7,8,9]), 128, shuffle=True, num_workers=2)
    test_A = DataLoader(get_class_subset(full_test, [0,1,2,3,4]), 128, num_workers=2)
    test_B = DataLoader(get_class_subset(full_test, [5,6,7,8,9]), 128, num_workers=2)

    # --- PHASE 1: SUPER-CHARGED BASE TRAINING ---
    print("\nPhase 1: High-Capacity Base Training (100 Epochs + MixUp + EMA)...")
    model = KFN(5).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    ema = EMA(model, decay=0.999)
    
    for ep in range(100): 
        model.train()
        for x, y in tqdm(loader_A, desc=f"Base Ep {ep+1}", leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            inputs, targets_a, targets_b, lam = mixup_data(x, y, alpha=0.4)
            logits, _ = model(inputs)
            loss = mixup_criterion(criterion, logits, targets_a, targets_b, lam)
            loss.backward()
            opt.step()
            ema.update(model)
        sched.step()
    
    ema.apply_shadow(model)
    acc_A = evaluate(model, test_A, device)
    print(f"--> Base Task A Acc (EMA): {acc_A:.2f}%") 
    
    # --- PREPARE EXPANSION ---
    print("\nPreparing Expansion...")
    teacher = copy.deepcopy(model)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False
    
    # REPLAY (Batch size 32 for Safety)
    replay_data = build_replay_buffer(full_train, [0,1,2,3,4], per_class=1000)
    replay_loader = DataLoader(replay_data, 32, shuffle=True) # SAFETY TWEAK 1
    replay_iter = iter(replay_loader)
    
    # Pre-train Specialist
    print("Pre-training New Specialist...")
    new_spec = Specialist().to(device)
    temp_head = nn.Linear(64, 5).to(device)
    spec_opt = torch.optim.AdamW(list(new_spec.parameters()) + list(temp_head.parameters()), lr=1e-3)
    spec_crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    for ep in range(25): 
        new_spec.train()
        for x, y in tqdm(loader_B, desc=f"Spec Ep {ep+1}", leave=False):
            x, y = x.to(device), (y - 5).to(device)
            spec_opt.zero_grad()
            feats = new_spec(x)
            pooled = F.adaptive_avg_pool2d(feats, 1).flatten(1)
            loss = spec_crit(temp_head(pooled), y)
            loss.backward()
            spec_opt.step()
            
    # Integrate
    print("Integrating...")
    model.specialists.append(new_spec)
    old_b = model.broker
    new_b = Broker(3, 64, 10).to(device)
    
    new_b.projections[0].load_state_dict(old_b.projections[0].state_dict())
    new_b.projections[1].load_state_dict(old_b.projections[1].state_dict())
    new_b.spatial_gate.load_state_dict(old_b.spatial_gate.state_dict())
    new_b.classifier = expand_classifier(old_b.classifier, 10).to(device)
    new_b.bottleneck[0] = expand_conv_layer(old_b.bottleneck[0], 192).to(device)
    new_b.channel_gate = expand_channel_gate(old_b.channel_gate, 2, 1, 64).to(device)
    model.broker = new_b
    
    # Partial Freeze (Deep Active)
    for i in [0, 1]:
        spec = model.specialists[i]
        for p in spec.conv1.parameters(): p.requires_grad = False
        for p in spec.bn1.parameters(): p.requires_grad = False
        for p in spec.layer1.parameters(): p.requires_grad = False
        # Unfreeze Deep
        for p in spec.layer2.parameters(): p.requires_grad = True
        for p in spec.layer3.parameters(): p.requires_grad = True
        spec.eval() 

    # --- PHASE 2: SAFE PLATINUM TRAINING ---
    print("Phase 2: Hybrid Training (50 Epochs + Safe Params)...")
    
    param_groups = [
        {'params': model.specialists[2].parameters(), 'lr': 1e-3}, 
        {'params': model.broker.parameters(), 'lr': 1e-4},         
        {'params': model.specialists[0].layer2.parameters(), 'lr': 1e-5}, 
        {'params': model.specialists[0].layer3.parameters(), 'lr': 1e-5},
        {'params': model.specialists[1].layer2.parameters(), 'lr': 1e-5},
        {'params': model.specialists[1].layer3.parameters(), 'lr': 1e-5},
    ]
    opt = torch.optim.AdamW(param_groups)
    
    ALPHA = 1.0 
    BETA = 1.5  # SAFETY TWEAK 2 (Reduced from 2.0 to help Task B)
    
    for ep in range(50): 
        model.train()
        model.specialists[0].eval()
        model.specialists[1].eval()
        
        for x_B, y_B in tqdm(loader_B, desc=f"Hybrid Ep {ep+1}", leave=False):
            x_B, y_B = x_B.to(device), y_B.to(device)
            
            try: x_A, y_A = next(replay_iter)
            except: 
                replay_iter = iter(replay_loader)
                x_A, y_A = next(replay_iter)
            x_A, y_A = x_A.to(device), y_A.to(device)
            
            opt.zero_grad()
            
            # Forward B
            logits_B, feats_B = model(x_B)
            loss_B = criterion(logits_B, y_B)
            
            # Forward A (Replay + Distill)
            logits_A, proj_A = model(x_A)
            with torch.no_grad():
                logits_T, proj_T = teacher(x_A)
            
            loss_replay = criterion(logits_A, y_A)
            loss_logit = F.kl_div(F.log_softmax(logits_A[:, :5]/2.0, dim=1), 
                                  F.softmax(logits_T[:, :5]/2.0, dim=1), 
                                  reduction='batchmean') * 4.0
            
            loss_feat = cosine_distill_loss(proj_A, proj_T)
            
            loss = loss_B + loss_replay + ALPHA*loss_logit + BETA*loss_feat
            
            loss.backward()
            opt.step()
            
    # --- FINAL VERDICT ---
    acc_A_final = evaluate(model, test_A, device)
    acc_B_final = evaluate(model, test_B, device)
    
    print("\n" + "="*40)
    print("FINAL RESULTS (SAFE PLATINUM)")
    print("="*40)
    print(f"Task A (Old): {acc_A:.2f}% -> {acc_A_final:.2f}%")
    print(f"Task B (New): {acc_B_final:.2f}%")
    print(f"Retention: {acc_A_final/acc_A:.1%}")
    print("="*40)

def evaluate(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

if __name__ == "__main__":
    run()

100%|██████████| 170M/170M [00:04<00:00, 35.3MB/s] 



Phase 1: High-Capacity Base Training (100 Epochs + MixUp + EMA)...


--> Base Task A Acc (EMA): 93.68%

Preparing Expansion...
Pre-training New Specialist...


Integrating...
Phase 2: Hybrid Training (50 Epochs + Safe Params)...



FINAL RESULTS (SAFE PLATINUM)
Task A (Old): 93.68% -> 93.42%
Task B (New): 16.98%
Retention: 99.7%
